# Llama3 Scene-Batch Validation on test_sent_emo.csv

This notebook sends one full dialogue scene per request to your Vertex endpoint using only Llama (no supporting agents), then writes utterance-level predictions to CSV in `logs/llama3/validation20_train`.

In [17]:
import os
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Optional, Tuple

import pandas as pd
import vertexai
from dotenv import load_dotenv
from sklearn.metrics import classification_report, f1_score
from vertexai.generative_models import GenerativeModel

load_dotenv()

PROJECT_ID = os.getenv("LLAMA_MODEL_PROJECT_ID") or os.getenv("TUNED_MODEL_PROJECT_ID")
LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
ENDPOINT_ID = "2346569469662330880"

if not PROJECT_ID:
    raise ValueError("Missing project id. Set LLAMA_MODEL_PROJECT_ID or TUNED_MODEL_PROJECT_ID in .env")

vertexai.init(project=PROJECT_ID, location=LOCATION)
llama3_model = GenerativeModel(f"projects/{PROJECT_ID}/locations/{LOCATION}/endpoints/{ENDPOINT_ID}")

print(f"Initialized endpoint: {ENDPOINT_ID}")

Initialized endpoint: 2346569469662330880


In [18]:
DEFAULT_INSTRUCTION = (
    """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert Emotion Recognition assistant specialized in the MELD benchmark.
For each scene, predict one emotion for EVERY utterance using labels: [anger, disgust, fear, joy, neutral, sadness, surprise].
Return JSON only in this schema:
{
  "predictions": [
    {"utterance_id": "id", "predicted_emotion": "label", "reasoning": "short reason"}
  ]
}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
)

def call_llama3(prompt: str) -> str:
    response = llama3_model.generate_content(prompt)
    return response.text

def parse_scene_predictions(text: str) -> dict:
    """Return mapping: utterance_id(str) -> {predicted_emotion, reasoning}."""
    if not text:
        return {}

    parsed = None
    try:
        match = re.search(r"\{[\s\S]*\}", text)
        if match:
            parsed = json.loads(match.group(0))
    except Exception:
        parsed = None

    if parsed is None:
        return {}

    # Supports both {"predictions": [...]} and direct list output
    predictions = []
    if isinstance(parsed, dict):
        maybe_list = parsed.get("predictions", [])
        if isinstance(maybe_list, list):
            predictions = maybe_list
    elif isinstance(parsed, list):
        predictions = parsed

    out = {}
    for item in predictions:
        if not isinstance(item, dict):
            continue
        utt_id = item.get("utterance_id")
        if utt_id is None:
            continue
        emotion = item.get("predicted_emotion") or item.get("emotion")
        reason = item.get("reasoning", "")
        out[str(utt_id)] = {
            "predicted_emotion": emotion.lower().strip() if isinstance(emotion, str) else None,
            "reasoning": str(reason) if reason is not None else ""
        }
    return out

def load_test_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.sort_values(["Dialogue_ID", "Utterance_ID"]).reset_index(drop=True)
    return df

def build_scene_batches_from_csv(df: pd.DataFrame) -> list:
    scenes = []

    for dialogue_id, group in df.groupby("Dialogue_ID", sort=False):
        scene_rows = []
        scene_lines = []

        for _, row in group.iterrows():
            utterance_id = str(row.get("Utterance_ID", "NA"))
            speaker = str(row.get("Speaker", "Unknown"))
            utterance = str(row.get("Utterance", ""))
            gt_emotion = str(row.get("Emotion", "")).strip().lower()
            gt_sentiment = str(row.get("Sentiment", "")).strip().lower()

            scene_rows.append({
                "Dialogue_ID": dialogue_id,
                "Utterance_ID": utterance_id,
                "Recognition_ID": f"{dialogue_id}_{utterance_id}",
                "Speaker": speaker,
                "Utterance": utterance,
                "ground_truth_completion": f"emotion: {gt_emotion}, sentiment: {gt_sentiment}",
                "ground_truth_emotion": gt_emotion,
                "ground_truth_sentiment": gt_sentiment,
            })
            scene_lines.append(f"{utterance_id} | {speaker}: {utterance}")

        scene_prompt = (
            f"{DEFAULT_INSTRUCTION}\n\n"
            f"Scene Dialogue ID: {dialogue_id}\n"
            "Utterances in order:\n"
            + "\n".join(scene_lines)
            + "\n\nReturn predictions for ALL utterance_id values above."
        )

        scenes.append({
            "dialogue_id": dialogue_id,
            "rows": scene_rows,
            "prompt": scene_prompt
        })

    return scenes

In [20]:
def run_validation(input_path: Path, output_dir: Path, limit: Optional[int] = None):
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pred_csv_path = output_dir / f"llama_only_scene_batch_predictions_test{timestamp}.csv"

    source_df = load_test_csv(input_path)
    scenes = build_scene_batches_from_csv(source_df)
    if limit is not None:
        scenes = scenes[:limit]

    total_scenes = len(scenes)
    total_utterances = sum(len(s["rows"]) for s in scenes)
    utterances_done = 0
    records = []

    for scene_idx, scene in enumerate(scenes, start=1):
        model_output = call_llama3(scene["prompt"])
        pred_map = parse_scene_predictions(model_output)

        for row in scene["rows"]:
            pred_item = pred_map.get(str(row["Utterance_ID"]), {})
            records.append({
                "Dialogue_ID": row["Dialogue_ID"],
                "Utterance_ID": row["Utterance_ID"],
                "Recognition_ID": row["Recognition_ID"],
                "Speaker": row["Speaker"],
                "Utterance": row["Utterance"],
                "ground_truth_completion": row["ground_truth_completion"],
                "ground_truth_emotion": row["ground_truth_emotion"],
                "ground_truth_sentiment": row["ground_truth_sentiment"],
                "predicted_emotion": pred_item.get("predicted_emotion"),
                "reasoning": pred_item.get("reasoning", ""),
                "scene_raw_output": model_output
            })

        utterances_done += len(scene["rows"])
        print(f"Progress: scenes {scene_idx}/{total_scenes} | utterances {utterances_done}/{total_utterances}")

    df = pd.DataFrame(records)
    df.to_csv(pred_csv_path, index=False)

    filled_predictions = int(df["predicted_emotion"].notna().sum()) if total_utterances else 0
    print(f"Total scenes sent to Llama: {total_scenes}")
    print(f"Total utterances covered: {total_utterances}")
    print(f"Rows with parsed predicted_emotion: {filled_predictions}")
    print(f"Predictions CSV saved to: {pred_csv_path}")

    valid_emotion_df = df.dropna(subset=["ground_truth_emotion", "predicted_emotion"]).copy()
    if not valid_emotion_df.empty:
        labels = sorted(valid_emotion_df["ground_truth_emotion"].dropna().unique().tolist())
        print("\nEmotion Classification Report")
        print(classification_report(
            valid_emotion_df["ground_truth_emotion"],
            valid_emotion_df["predicted_emotion"],
            labels=labels,
            digits=4,
            zero_division=0
        ))
        print(
            "Weighted F1:",
            round(
                float(f1_score(
                    valid_emotion_df["ground_truth_emotion"],
                    valid_emotion_df["predicted_emotion"],
                    average="weighted",
                    zero_division=0
                )),
                4
            )
        )

    return pred_csv_path, df

In [22]:
input_path = Path("../data/test_sent_emo.csv")
output_dir = Path("../logs/llama3/validation20_train")
limit = None  # Full scene split

csv_path, preview_df = run_validation(
    input_path=input_path,
    output_dir=output_dir,
    limit=limit
)

print(f"Final predictions CSV: {csv_path}")
preview_df.head()

Progress: scenes 1/280 | utterances 3/2610
Progress: scenes 2/280 | utterances 11/2610
Progress: scenes 3/280 | utterances 22/2610
Progress: scenes 4/280 | utterances 29/2610
Progress: scenes 5/280 | utterances 38/2610
Progress: scenes 6/280 | utterances 47/2610
Progress: scenes 7/280 | utterances 50/2610
Progress: scenes 8/280 | utterances 59/2610
Progress: scenes 9/280 | utterances 66/2610
Progress: scenes 10/280 | utterances 84/2610
Progress: scenes 11/280 | utterances 88/2610
Progress: scenes 12/280 | utterances 93/2610
Progress: scenes 13/280 | utterances 113/2610
Progress: scenes 14/280 | utterances 121/2610
Progress: scenes 15/280 | utterances 132/2610
Progress: scenes 16/280 | utterances 138/2610
Progress: scenes 17/280 | utterances 148/2610
Progress: scenes 18/280 | utterances 181/2610
Progress: scenes 19/280 | utterances 184/2610
Progress: scenes 20/280 | utterances 185/2610
Progress: scenes 21/280 | utterances 193/2610
Progress: scenes 22/280 | utterances 200/2610
Progress: 

,Dialogue_ID,Utterance_ID,Recognition_ID,Speaker,Utterance,ground_truth_completion,ground_truth_emotion,ground_truth_sentiment,predicted_emotion,reasoning,scene_raw_output
0,0,0,0_0,Mark,Why do all youre coffee mugs have numbers on ...,"emotion: surprise, sentiment: positive",surprise,positive,neutral,"Mark is asking a neutral, informational questi...","```json\n{\n ""predictions"": [\n {\n ""..."
1,0,1,0_1,Rachel,Oh. Thats so Monica can keep track. That way ...,"emotion: anger, sentiment: negative",anger,negative,neutral,"Rachel is providing a neutral, factual explana...","```json\n{\n ""predictions"": [\n {\n ""..."
2,0,2,0_2,Rachel,Y'know what?,"emotion: neutral, sentiment: neutral",neutral,neutral,neutral,"Rachel is making a neutral, preparatory statem...","```json\n{\n ""predictions"": [\n {\n ""..."
3,1,0,1_0,Joey,"Come on, Lydia, you can do it.","emotion: neutral, sentiment: neutral",neutral,neutral,neutral,Neutral statement of encouragement without str...,"```json\n{\n ""predictions"": [\n {\n ""..."
4,1,1,1_1,Joey,Push!,"emotion: joy, sentiment: positive",joy,positive,neutral,"Neutral, an imperative command without clear e...","```json\n{\n ""predictions"": [\n {\n ""..."


In [21]:
# Optional quick check before full run
tmp_df = load_test_csv(Path("../data/test_sent_emo.csv"))
tmp_scenes = build_scene_batches_from_csv(tmp_df)
tmp_utterances = sum(len(s["rows"]) for s in tmp_scenes)
print(f"Scenes prepared for llama inference: {len(tmp_scenes)}")
print(f"Utterances covered: {tmp_utterances}")
print("Run the previous cell to execute full scene-batch inference and write CSV.")

Scenes prepared for llama inference: 280
Utterances covered: 2610
Run the previous cell to execute full scene-batch inference and write CSV.


In [31]:
from sklearn.metrics import classification_report, f1_score
import pandas as pd

VALID_EMOTIONS = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}

def normalize_emotion_label(value):
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    if label in VALID_EMOTIONS:
        return label
    return None

results_df = pd.read_csv("../logs/llama3/validation20_train/llama_only_scene_batch_predictions_test20260419_203932.csv")

eval_df = results_df[["ground_truth_emotion", "predicted_emotion"]].copy()
eval_df["ground_truth_emotion"] = eval_df["ground_truth_emotion"].apply(normalize_emotion_label)
eval_df["predicted_emotion"] = eval_df["predicted_emotion"].apply(normalize_emotion_label)
eval_df = eval_df.dropna(subset=["ground_truth_emotion", "predicted_emotion"])

y_true = eval_df["ground_truth_emotion"].astype(str).tolist()
y_pred = eval_df["predicted_emotion"].astype(str).tolist()
labels = sorted(VALID_EMOTIONS)

classification_report_llama = classification_report(
    y_true=y_true,
    y_pred=y_pred,
    labels=labels,
    digits=4,
    zero_division=0
)
weighted_f1 = f1_score(
    y_true=y_true,
    y_pred=y_pred,
    average="weighted",
    zero_division=0
)

print(f"Evaluated rows: {len(eval_df)} / {len(results_df)}")
print("Llama3 Emotion Classification Report:")
print(classification_report_llama)
print(f"Weighted F1: {weighted_f1:.4f}")

Evaluated rows: 2474 / 2610
Llama3 Emotion Classification Report:
              precision    recall  f1-score   support

       anger     0.9471    0.5934    0.7296       332
     disgust     0.8696    0.5970    0.7080        67
        fear     0.6667    0.6400    0.6531        50
         joy     0.8840    0.5785    0.6994       382
     neutral     0.7451    0.9721    0.8436      1182
     sadness     0.7128    0.7020    0.7074       198
    surprise     0.8324    0.5856    0.6875       263

    accuracy                         0.7809      2474
   macro avg     0.8082    0.6669    0.7184      2474
weighted avg     0.8022    0.7809    0.7710      2474

Weighted F1: 0.7710
